# T1-07 Data classification

**Time:** 09:20–10:30 &nbsp;|&nbsp; **Evidence:** classification score on CI-01…CI-10

**Notebook learning outcomes**

- Say what an **index** is, in CivicDesk words.
- Label a piece of text **INDEX** or **NEVER**.
- Explain a **canary**, and why clearance is not the same as “allowed in the index.”


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the teaching note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** run all cells. Change only lines marked `# [CHANGE]`.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path. Do not drive someone else's notebook.

Git is not required. If you are stuck, load the matching file in `completed/` for this notebook only, mark it as loaded from reference, and continue.


## Overnight readers — the new words

Yesterday the model only saw what you typed. Today you will give it a **corpus**: a pile of approved CivicDesk paragraphs. Before you pile them, you classify.

| Word | Plain meaning | CivicDesk example |
|---|---|---|
| **Corpus** | The set of documents you are willing to search | Twelve fictional handbook chunks, not an EDMS export |
| **Chunk** | One retrieval unit (a paragraph or short section), not a 200-page PDF | `HB-VAC-C2` is one chunk |
| **Index** | The searchable structure over chunks (keyword overlap here; embeddings in production) | Only chunks with `index_ok: true` |
| **Metadata** | Labels on a chunk that are not the paragraph itself | `clearance`, `id`, `citation` |
| **Canary** | Planted sensitive text. If a reply quotes it, the exercise **fails** | National ID `999-00-1111`, name Maya Reid |
| **PII** | Information that identifies a person | CVs, burgess forms, health records, live tickets |

**INDEX** means: this text may be stored in the classroom search pile. **NEVER** means: do not copy it in, even “just to try.”

This is **N5** (data ban) and the start of **N4** (access). T1-10 is the access gate. Today is “does it enter the pile at all?”

![May this enter the CivicDesk index?](images/t2_index_or_not.jpg)


## Ministry link

| Your EOI type | What must never enter an index in *this* room |
|---|---|
| **Labour** | Real CVs, scores, names |
| **Health / THA** | Patient detail, LMS submissions, suicide-guideline *cases* (the *rule* N1 may be indexed) |
| **Finance** | Cabinet *bodies*. Class metadata is a later access question |
| **RDLG** | Burgess applications, household names, addresses |
| **TTDF** | Patrol notes, landing sites, tracks (AUX-05) |
| **Trade / Defence assets** | Live GLPI/serial dumps — query the register; do not index this month’s extract |
| **all** | The canary line. If you index it, you fail the drill |

**N14 woven:** someone must be able to say later “this chunk was / was not in the index.” That is why every chunk has an id.


## 1. Look at the handbook pile (including the canary)

`index_ok` is a flag on the chunk. `clearance` is who may *see* it later. A restricted chunk can still be in the index (T1-10). A prohibited canary must not be.


In [ ]:
# [RUN]
import civicdesk_rag as rag

print("All chunks in data/handbooks.json\n")
rag.print_chunks(include_prohibited=True)
print()
print("Indexed only (index_ok True):", len(rag.indexed_chunks()), "of", len(rag.load_chunks()))


### Teaching note

`HB-CANARY` has `index_ok: false` and `clearance: prohibited`. The helper `retrieve()` skips it unless you pass `include_prohibited=True` (a drill, not a product feature).

`HB-CAB-X` is **restricted** but `index_ok: true`. That is deliberate. The index may know a classed minute exists. Displaying the body is T1-10. Finance-shaped: even that is generous — many rooms would keep Cabinet bodies **out** of the index entirely (see CI-05).


## 2. Classification sheet

Read CI-01…CI-10. Decide INDEX or NEVER **before** you open the next cell if you can.


In [ ]:
# [RUN]
import civicdesk_rag as rag

rag.print_classify_sheet()


## 3. Score your labels

Allowed values: `index` or `never`. The cell ships the teaching key so Run All works. Cover it and try first if you want a real score.


In [ ]:
# [CHANGE] replace with your labels to test yourself.
GUESSES = {
    "CI-01": "index",
    "CI-02": "never",
    "CI-03": "index",
    "CI-04": "never",
    "CI-05": "never",
    "CI-06": "index",
    "CI-07": "never",
    "CI-08": "index",
    "CI-09": "never",
    "CI-10": "never",
}

import civicdesk_rag as rag

rag.score_classification(GUESSES)
rag.save_log("T1-07_classify", {"guesses": GUESSES})


### After you run

A published **rule** can be indexed. A **person’s file** cannot. A **live extract** of a register cannot — you would be copying operational data into a search pile that the model might quote.

**Write-up:** name one thing from *your* overnight canvas that would be NEVER, in CivicDesk words (no real identifiers).


In [ ]:
# [OPTIONAL] what retrieve() would do if someone left the canary searchable
import civicdesk_rag as rag

print("Normal retrieve (canary excluded):")
for h in rag.retrieve("Maya Reid national ID", k=5):
    print(" ", h["id"], "score", h["score"])
print()
print("Unsafe retrieve (include_prohibited=True) — classroom drill only:")
for h in rag.retrieve("Maya Reid national ID", k=5, include_prohibited=True):
    print(" ", h["id"], "score", h["score"], "index_ok", h.get("index_ok"))
